<a href="https://colab.research.google.com/github/pranavvup-byte/ml-project/blob/main/projectcdk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [4]:
df = pd.read_csv('/content/sample_data/CKD_NHANES_2021_2023.csv')

print("Dataset Shape:", df.shape)
print(df.head())

Dataset Shape: (11933, 29)
   participant_id   age  gender           ethnicity  education_level  \
0        130378.0  43.0    Male  Non-Hispanic Asian              5.0   
1        130379.0  66.0    Male  Non-Hispanic White              5.0   
2        130380.0  44.0  Female      Other Hispanic              3.0   
3        130381.0   5.0  Female   Other/Multiracial              NaN   
4        130382.0   2.0    Male  Non-Hispanic White              NaN   

   poverty_income_ratio   bmi  weight_kg  height_cm  bp_systolic  ...  \
0                  5.00  27.0       86.9      179.5        135.0  ...   
1                  5.00  33.5      101.8      174.2        121.0  ...   
2                  1.41  29.7       69.4      152.9        111.0  ...   
3                  1.53  23.8       34.3      120.1          NaN  ...   
4                  3.60   NaN       13.6        NaN          NaN  ...   

   urine_albumin  albumin_creatinine_ratio  diabetes_diagnosed  insulin_use  \
0          23.12      

In [5]:
target = 'ckd_present'

X = df.drop(columns=[target])
y = df[target]

# Remove leakage / ID columns if they exist
remove_columns = ['participant_id', 'ckd_stage', 'eGFR']

for col in remove_columns:
    if col in X.columns:
        X = X.drop(columns=[col])

print("Features:", X.shape)
print("Target:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

Features: (11933, 26)
Target: (11933,)

Target distribution:
ckd_present
1    8341
0    3592
Name: count, dtype: int64


In [6]:
numeric_features = X.select_dtypes(
    include=['int64', 'float64', 'int32', 'float32']
).columns.tolist()

categorical_features = X.select_dtypes(
    include=['object', 'category', 'bool']
).columns.tolist()

print("Numerical columns:")
print(numeric_features)

print("\nCategorical columns:")
print(categorical_features)


Numerical columns:
['age', 'education_level', 'poverty_income_ratio', 'bmi', 'weight_kg', 'height_cm', 'bp_systolic', 'bp_diastolic', 'serum_creatinine', 'blood_urea_nitrogen', 'albumin_serum', 'phosphorus', 'bicarbonate', 'calcium', 'uric_acid', 'urine_creatinine', 'urine_albumin', 'albumin_creatinine_ratio', 'diabetes_diagnosed', 'insulin_use', 'diabetes_pills', 'ever_smoked', 'current_smoker', 'egfr']

Categorical columns:
['gender', 'ethnicity']


In [7]:
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (9546, 26)
Testing data: (2387, 26)


In [10]:
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    random_state=42,
    eval_metric='logloss'
)

xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', xgb)
])

xgb_pipeline.fit(X_train, y_train)

print("XGBoost model trained successfully!")

XGBoost model trained successfully!


In [11]:
y_pred_xgb = xgb_pipeline.predict(X_test)
y_prob_xgb = xgb_pipeline.predict_proba(X_test)[:, 1]

print("XGBoost Results")
print("----------------")

print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("Precision:", precision_score(y_test, y_pred_xgb))
print("Recall:", recall_score(y_test, y_pred_xgb))
print("F1 Score:", f1_score(y_test, y_pred_xgb))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_xgb))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb))

XGBoost Results
----------------
Accuracy: 0.9991621281943862
Precision: 0.999400479616307
Recall: 0.999400479616307
F1 Score: 0.999400479616307
ROC-AUC: 0.9999974985241292

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       719
           1       1.00      1.00      1.00      1668

    accuracy                           1.00      2387
   macro avg       1.00      1.00      1.00      2387
weighted avg       1.00      1.00      1.00      2387



In [12]:
 lgbm = LGBMClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    verbosity=-1
)

lgbm_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', lgbm)
])

lgbm_pipeline.fit(X_train, y_train)

print("LightGBM model trained successfully!")


LightGBM model trained successfully!


In [13]:
y_pred_lgbm = lgbm_pipeline.predict(X_test)
y_prob_lgbm = lgbm_pipeline.predict_proba(X_test)[:, 1]

print("LightGBM Results")
print("--------------------")

print("Accuracy :", accuracy_score(y_test, y_pred_lgbm))
print("Precision:", precision_score(y_test, y_pred_lgbm))
print("Recall   :", recall_score(y_test, y_pred_lgbm))
print("F1 Score :", f1_score(y_test, y_pred_lgbm))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob_lgbm))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_lgbm))


LightGBM Results
--------------------
Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1 Score : 1.0
ROC-AUC  : 1.0

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       719
           1       1.00      1.00      1.00      1668

    accuracy                           1.00      2387
   macro avg       1.00      1.00      1.00      2387
weighted avg       1.00      1.00      1.00      2387



/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [14]:
comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC'],
    'XGBoost': [
        accuracy_score(y_test, y_pred_xgb),
        precision_score(y_test, y_pred_xgb),
        recall_score(y_test, y_pred_xgb),
        f1_score(y_test, y_pred_xgb),
        roc_auc_score(y_test, y_prob_xgb)
    ],
    'LightGBM': [
        accuracy_score(y_test, y_pred_lgbm),
        precision_score(y_test, y_pred_lgbm),
        recall_score(y_test, y_pred_lgbm),
        f1_score(y_test, y_pred_lgbm),
        roc_auc_score(y_test, y_prob_lgbm)
    ]
})

print(comparison)

      Metric   XGBoost  LightGBM
0   Accuracy  0.999162       1.0
1  Precision  0.999400       1.0
2     Recall  0.999400       1.0
3   F1 Score  0.999400       1.0
4    ROC-AUC  0.999997       1.0
